# CHiRPE Quick Start Guide

This notebook demonstrates how to use CHiRPE for Clinical High-Risk for Psychosis (CHR-P) prediction.

## Overview

1. Load and preprocess data
2. Train a classifier
3. Make predictions
4. Generate explanations

In [ ]:
# Setup paths
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

## 1. Generate Synthetic Data

First, let's generate some synthetic PSYCHS interview data for testing.

In [ ]:
from scripts.generate_synthetic_data import generate_synthetic_dataset
from chirpe.data.dataset import split_data

# Generate 100 synthetic transcripts
data = generate_synthetic_dataset(
    n_participants=100,
    chr_ratio=0.836,  # Match paper's distribution
    seed=42
)

print(f"Generated {len(data)} transcripts")

# Split into train/val/test
train_data, val_data, test_data = split_data(
    data, 
    train_ratio=0.64,
    val_ratio=0.16,
    seed=42
)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

## 2. Preprocess Data

Segment transcripts by symptom domain and summarize.

In [ ]:
from chirpe.data.preprocessor import TranscriptPreprocessor

# Initialize preprocessor
preprocessor = TranscriptPreprocessor(
    segmentation_threshold=0.80,
    use_llm_summarizer=False  # Use simple summarizer for demo
)

# Process a sample
sample = train_data[0]
processed = preprocessor.process_transcript(
    sample['transcript'],
    sample['participant_id']
)

print(f"Participant: {processed['participant_id']}")
print(f"Label: {sample['label']}")
print(f"Number of segments: {processed['num_segments']}")
print(f"\nDomains covered: {processed['domains_covered']}")

In [ ]:
# View a segment summary
if processed['segments']:
    seg = processed['segments'][0]
    print(f"Domain: {seg['domain']}")
    print(f"\nOriginal text: {seg['text'][:200]}...")
    print(f"\nSummary: {seg['summary']}")

## 3. Train a Classifier

Train a BERT-based classifier on the processed data.

In [ ]:
from chirpe.models.classifier import CHRClassifier
from chirpe.models.trainer import ModelTrainer
from chirpe.data.dataset import CHRPDataset

# Prepare data with summaries
def prepare_data(data, preprocessor):
    prepared = []
    for item in data:
        processed = preprocessor.process_transcript(
            item['transcript'],
            item['participant_id']
        )
        # Concatenate all summaries
        summary_text = ' '.join([s['summary'] for s in processed['segments']])
        prepared.append({
            'participant_id': item['participant_id'],
            'summary': summary_text,
            'label': item['label']
        })
    return prepared

train_prepared = prepare_data(train_data[:50], preprocessor)  # Use subset for demo
val_prepared = prepare_data(val_data[:10], preprocessor)

print(f"Prepared {len(train_prepared)} training samples")

In [ ]:
# Initialize classifier
classifier = CHRClassifier(
    model_name='bert-base-uncased',
    num_labels=2
)

# Create datasets
train_dataset = CHRPDataset(
    train_prepared,
    classifier.tokenizer,
    max_length=512
)

val_dataset = CHRPDataset(
    val_prepared,
    classifier.tokenizer,
    max_length=512
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

In [ ]:
# Train (using minimal epochs for demo)
trainer = ModelTrainer(
    classifier=classifier,
    output_dir='../outputs/demo',
    learning_rate=2e-5,
    batch_size=4,
    num_epochs=1,  # Use 1 epoch for demo
    weight_decay=0.01
)

trainer.train(train_dataset, val_dataset)
print("Training complete!")

## 4. Make Predictions

Test the trained model on new data.

In [ ]:
# Test on a sample
test_sample = test_data[0]
test_processed = preprocessor.process_transcript(
    test_sample['transcript'],
    test_sample['participant_id']
)

test_segments = test_processed['segments']
print(f"Number of segments: {len(test_segments)}")

# Predict with segments
results = classifier.predict_with_segments(test_segments)

print(f"\nTrue label: {test_sample['label']}")
print(f"Prediction: {'CHR-P' if results['prediction'] == 1 else 'Healthy'}")
print(f"Confidence: {results['confidence']:.4f}")
print(f"Number of segments analyzed: {results['num_segments']}")

## 5. Generate SHAP Explanations

Create clinician-friendly explanations for predictions.

In [ ]:
from chirpe.explanations.shap_generator import SHAPExplainer

# Initialize explainer
explainer = SHAPExplainer(
    classifier.model,
    classifier.tokenizer,
    device=classifier.device
)

# Get explanations for segments
explanations = explainer.explain_segments(test_segments)

print(f"Generated explanations for {len(explanations)} segments")
print(f"\nDomains: {list(explanations.keys())}")

In [ ]:
# Get word-level summary for first segment
first_domain = list(explanations.keys())[0]
first_exp = explanations[first_domain]

top_words = explainer.get_word_level_summary(first_exp, top_k=10)
print(f"Top contributing words for {first_domain}:")
for word, score in top_words:
    print(f"  {word}: {score:.4f}")

In [ ]:
# Aggregate by domain
domain_scores = explainer.aggregate_by_domain(explanations)

print("\nDomain SHAP scores:")
for domain, score in sorted(domain_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {domain}: {score:.4f}")

In [ ]:
# Generate narrative summaries
from chirpe.explanations.narrative import SimpleNarrativeGenerator

narrative_gen = SimpleNarrativeGenerator()
narratives = narrative_gen.generate_all_narratives(test_segments, top_k=3)

print("Narrative Summaries:")
for i, narrative in enumerate(narratives, 1):
    print(f"\n{i}. {narrative['symptom']}:")
    print(f"   {narrative['narrative']}")

## 6. Visualizations

Create visualization plots.

In [ ]:
import matplotlib.pyplot as plt

# Plot word-level SHAP values
fig = explainer.plot_word_level(first_exp, top_k=10)
plt.title(f"Word-level SHAP: {first_domain}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot symptom-level SHAP values
fig = explainer.plot_symptom_level(domain_scores)
plt.title("Symptom-Level SHAP Values")
plt.tight_layout()
plt.show()

## 7. Evaluation

Evaluate model performance on test set.

In [ ]:
from chirpe.data.dataset import CHRPDataset
from chirpe.models.trainer import ModelTrainer

# Prepare test data
test_prepared = prepare_data(test_data[:20], preprocessor)
test_dataset = CHRPDataset(test_prepared, classifier.tokenizer)

# Evaluate
results = trainer.evaluate(test_dataset)

print("\nTest Results:")
for metric, value in results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")

## Summary

This notebook demonstrated:

1. **Data Generation**: Synthetic PSYCHS interview data
2. **Preprocessing**: Symptom segmentation and summarization
3. **Training**: BERT-based classifier
4. **Prediction**: Segment-level voting for transcript classification
5. **Explanations**: SHAP-based word-level, symptom-level, and narrative explanations
6. **Evaluation**: Model performance metrics

For production use:
- Train on real PSYCHS transcript data
- Use LLM-based summarization for better quality
- Fine-tune hyperparameters via cross-validation
- Evaluate with clinical expert feedback